# Clinical Data Ensembles: XGBoost, LGBoost, RFC.
## Run these models on Cleveland clinical data, and test on combined clinical data. Report: AUC >= 0.85, Acc >= 0.80, F1 >= 0.80. \\
Hyperparameter tuning is not required, do it during final modeling.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.metrics import roc_auc_score

In [ ]:
df_clev = pd.read_csv('/content/sample_data/cleveland.csv')
df_comb = pd.read_csv('/content/combined_statslogExcluded.csv')

X_train = df_clev.drop('target',axis=1)
y_train = df_clev['target']

X_test = df_comb.drop('target',axis=1)
y_test = df_comb['target']


In [ ]:
from scipy.stats import t

def get_ci(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    se = np.std(data, ddof=1) / np.sqrt(n)  # Standard error
    h = t.ppf((1 + confidence) / 2., n-1) * se
    return mean, mean - h, mean + h

Applying XGBoost

In [ ]:
import xgboost as xgb


xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False
)

xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Get probabilities for the positive class
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print("AUC Score:", auc)



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:55:03] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.82      0.72      0.77       410
           1       0.79      0.87      0.83       508

    accuracy                           0.80       918
   macro avg       0.81      0.80      0.80       918
weighted avg       0.80      0.80      0.80       918

AUC Score: 0.8853730555022086


Getting 95% CI of accuracy, f1-score, and auc score

In [ ]:
# 95% CI of accuracy, f1-score, auc score
auc_read = []
acc_read = []
f1_read = []

for i in range(100):
    xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False)

    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

print(len(auc_read))
print(len(acc_read))
print(len(f1_read))

In [ ]:
auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")


AUC: 0.8854 [0.8854, 0.8854]
ACC: 0.8028 [0.8028, 0.8028]
F1 : 0.8297 [0.8297, 0.8297]


Modeling LGBoost

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
lgbm_clf = LGBMClassifier(verbose=-1)

# 2. Fit the model
lgbm_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = lgbm_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


📊 Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.72      0.78       410
           1       0.80      0.89      0.84       508

    accuracy                           0.82       918
   macro avg       0.82      0.81      0.81       918
weighted avg       0.82      0.82      0.81       918

AUC Score: 0.8891


Getting 95% CI of accuracy, f1-score, and auc score

In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    lgbm_clf = LGBMClassifier(verbose=-1)
    lgbm_clf.fit(X_train, y_train)
    y_pred = lgbm_clf.predict(X_test)
    y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))



In [ ]:
auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.8891 [0.8891, 0.8891]
ACC: 0.8159 [0.8159, 0.8159]
F1 : 0.8425 [0.8425, 0.8425]


Modeling Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
rfc_clf = RandomForestClassifier()

# 2. Fit the model
rfc_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = rfc_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = rfc_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


📊 Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.80      0.81       410
           1       0.84      0.85      0.85       508

    accuracy                           0.83       918
   macro avg       0.83      0.83      0.83       918
weighted avg       0.83      0.83      0.83       918

AUC Score: 0.9112


Getting 95% CI of accuracy, f1-score, and auc score

In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    rfc_clf = RandomForestClassifier()
    rfc_clf.fit(X_train, y_train)
    y_pred = rfc_clf.predict(X_test)
    y_proba = rfc_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.9102 [0.9094, 0.9110]
ACC: 0.8292 [0.8276, 0.8308]
F1 : 0.8499 [0.8483, 0.8515]


# Non-Clinical Data ensembles: XGBoost, LGBoost, RFC
## Run these models on Cleveland clinical data, and test on combined clinical data. Report: AUC >= 0.85, Acc >= 0.80, F1 >= 0.80. \\
Hyperparameter tuning is not required, do it during final modeling.

In [ ]:
df_clev.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,target
0,63,1,0,145,233,1,2,150,0,2.3,2,0
1,67,1,3,160,286,0,2,108,1,1.5,1,1
2,67,1,3,120,229,0,2,129,1,2.6,1,1
3,37,1,2,130,250,0,0,187,0,3.5,2,0
4,41,0,1,130,204,0,2,172,0,1.4,0,0


In [ ]:
df_clev_nonc = df_clev.drop(columns=['chol','restecg','slope', 'oldpeak'],axis=1)
df_comb_nonc = df_comb.drop(columns=['chol','restecg','slope', 'oldpeak'],axis=1)

In [ ]:
df_clev_nonc.head()

In [ ]:
df_comb_nonc.head()

In [ ]:
X_train = df_clev_nonc.drop('target',axis=1)
y_train = df_clev_nonc['target']

X_test = df_comb_nonc.drop('target',axis=1)
y_test = df_comb_nonc['target']



Modeling XGBoost

In [ ]:
import xgboost as xgb


xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False
)

xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Get probabilities for the positive class
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print("AUC Score:", auc)


              precision    recall  f1-score   support

           0       0.87      0.68      0.77       410
           1       0.78      0.92      0.85       508

    accuracy                           0.81       918
   macro avg       0.83      0.80      0.81       918
weighted avg       0.82      0.81      0.81       918

AUC Score: 0.8821034184751297


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:21:50] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# 95% CI of accuracy, f1-score, auc score
auc_read = []
acc_read = []
f1_read = []

for i in range(100):
    xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False)

    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:25] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

AUC: 0.8821 [0.8821, 0.8821]
ACC: 0.8137 [0.8137, 0.8137]
F1 : 0.8452 [0.8452, 0.8452]


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:22:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

Modeling LGBoost

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
lgbm_clf = LGBMClassifier(verbose=-1)

# 2. Fit the model
lgbm_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = lgbm_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


              precision    recall  f1-score   support

           0       0.86      0.67      0.75       410
           1       0.77      0.91      0.84       508

    accuracy                           0.80       918
   macro avg       0.82      0.79      0.79       918
weighted avg       0.81      0.80      0.80       918

AUC Score: 0.8729


In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    lgbm_clf = LGBMClassifier(verbose=-1)
    lgbm_clf.fit(X_train, y_train)
    y_pred = lgbm_clf.predict(X_test)
    y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))


auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.8729 [0.8729, 0.8729]
ACC: 0.8028 [0.8028, 0.8028]
F1 : 0.8365 [0.8365, 0.8365]


Modeling RFC

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
rfc_clf = RandomForestClassifier()

# 2. Fit the model
rfc_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = rfc_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = rfc_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


📊 Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.64      0.74       410
           1       0.76      0.92      0.83       508

    accuracy                           0.80       918
   macro avg       0.81      0.78      0.78       918
weighted avg       0.81      0.80      0.79       918

AUC Score: 0.8892


In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    rfc_clf = RandomForestClassifier()
    rfc_clf.fit(X_train, y_train)
    y_pred = rfc_clf.predict(X_test)
    y_proba = rfc_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.8905 [0.8899, 0.8911]
ACC: 0.8038 [0.8029, 0.8048]
F1 : 0.8381 [0.8373, 0.8389]


# Generalization Proved (Clinical): Use combined data for modeling.
## First shuffle the data. SMOTE it. Then, split into 80:20 train test. Train and test the three ensemble models. Report 95% CI of AUC, ACC, F1. Show classification report.  

In [ ]:
df_comb = pd.read_csv('/content/combined_statslogExcluded.csv')
df_comb.shape
df_comb.isna().sum()

In [ ]:
df_comb.head()

In [ ]:
from sklearn.utils import shuffle
df_comb = shuffle(df_comb)
df_comb.head()

In [ ]:
from imblearn.over_sampling import SMOTE

X = df_comb.drop('target', axis=1)
y = df_comb['target']

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

df_comb_resampled = pd.DataFrame(X_resampled, columns=X.columns)
df_comb_resampled['target'] = y_resampled

df_comb = df_comb_resampled.copy()

print(df_comb.shape)
print(df_comb['target'].value_counts())

(1016, 12)
target
1    508
0    508
Name: count, dtype: int64


In [ ]:
df_comb['target'].value_counts()

,count
target,
1,508
0,508


In [ ]:
df_comb.to_csv('combined_statslogExcluded_SMOTE.csv',index=False)

In [ ]:
df_comb = pd.read_csv('/content/combined_statslogExcluded_SMOTE.csv')

In [ ]:
from sklearn.model_selection import train_test_split
X = df_comb.drop('target',axis=1)
y = df_comb['target']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=123)

## XGBoost Modeling

In [ ]:
# Hyperparameter tuning
!pip install optuna
import optuna

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 1.0),
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "objective": "binary:logistic"
    }

    model = XGBClassifier(**params)
    score = cross_val_score(model, X_train, y_train, cv=10, scoring=make_scorer(f1_score)).mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)




In [ ]:
print("Best Hyperparameters:")
print(study.best_params)

Best Hyperparameters:
{'n_estimators': 115, 'max_depth': 6, 'learning_rate': 0.05477560713972631, 'subsample': 0.5715501998326323, 'colsample_bytree': 0.5422105156870113, 'gamma': 1.589464464224486, 'reg_alpha': 0.9968474868144839, 'reg_lambda': 0.8258515199190066}


In [ ]:
xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False,
    n_estimators = 115, max_depth = 6, learning_rate = 0.0547, subsample = 0.5715,
    colsample_bytree = 0.5422, gamma = 1.5894, reg_alpha = 0.9968, reg_lambda = 0.8258
)

xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Get probabilities for the positive class
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print("AUC Score:", auc)


              precision    recall  f1-score   support

           0       0.92      0.89      0.91       102
           1       0.90      0.92      0.91       102

    accuracy                           0.91       204
   macro avg       0.91      0.91      0.91       204
weighted avg       0.91      0.91      0.91       204

AUC Score: 0.9578046905036525


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:27:00] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# 95% CI of accuracy, f1-score, auc score
auc_read = []
acc_read = []
f1_read = []

for i in range(100):
    xgb_clf = xgb.XGBClassifier(objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False,
    n_estimators = 115, max_depth = 6, learning_rate = 0.0547, subsample = 0.5715,
    colsample_bytree = 0.5422, gamma = 1.5894, reg_alpha = 0.9968, reg_lambda = 0.8258)

    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:26] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

AUC: 0.9578 [0.9578, 0.9578]
ACC: 0.9069 [0.9069, 0.9069]
F1 : 0.9082 [0.9082, 0.9082]


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:31] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:31] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:31] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:28:31] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


## LGBoost Modeling

In [ ]:
# Hyperparameter tuning
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "num_leaves": trial.suggest_int("num_leaves", 15, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
    }

    model = LGBMClassifier(**params,verbose=-1)
    score = cross_val_score(model, X_train, y_train, cv=10, scoring=make_scorer(f1_score)).mean()
    return score

# Start Bayesian Pptimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)




In [ ]:
# Show best parameters and score from Bayesian Optimization
print("Best Hyperparameters for LGBM:")
print(study.best_params)

Best Hyperparameters for LGBM:
{'n_estimators': 125, 'max_depth': 3, 'learning_rate': 0.036018501079536275, 'subsample': 0.5064528141022786, 'colsample_bytree': 0.5004725650039892, 'num_leaves': 108, 'min_child_samples': 21, 'reg_alpha': 0.8659794592877946, 'reg_lambda': 0.6940468425190104}


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
lgbm_clf = LGBMClassifier(n_estimators=125, max_depth=3, learning_rate=0.0360, subsample=0.5064,
                          colsample_bytree=0.5004, num_leaves=108, min_child_samples=21, reg_alpha=0.8659,
                          reg_lambda=0.6940, verbose=-1)

# 2. Fit the model
lgbm_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = lgbm_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


              precision    recall  f1-score   support

           0       0.91      0.89      0.90       102
           1       0.89      0.91      0.90       102

    accuracy                           0.90       204
   macro avg       0.90      0.90      0.90       204
weighted avg       0.90      0.90      0.90       204

AUC Score: 0.9537


In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    lgbm_clf = LGBMClassifier(n_estimators=125, max_depth=3, learning_rate=0.0360, subsample=0.5064,
                          colsample_bytree=0.5004, num_leaves=108, min_child_samples=21, reg_alpha=0.8659,
                          reg_lambda=0.6940, verbose=-1)
    lgbm_clf.fit(X_train, y_train)
    y_pred = lgbm_clf.predict(X_test)
    y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))


auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.9537 [0.9537, 0.9537]
ACC: 0.9020 [0.9020, 0.9020]
F1 : 0.9029 [0.9029, 0.9029]


## RFC Modeling

In [ ]:
# hyperparameter tuning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, make_scorer

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
    }

    model = RandomForestClassifier(**params, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=10, scoring=make_scorer(f1_score)).mean()
    return score

# Start the study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)




In [ ]:
# Show results
print("Best Hyperparameters for RandomForest:")
print(study.best_params)

Best Hyperparameters for RandomForest:
{'n_estimators': 256, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True}


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
rfc_clf = RandomForestClassifier(n_estimators=256, max_depth=14, min_samples_split=7, min_samples_leaf=1,
                                 max_features='sqrt',bootstrap=True)

# 2. Fit the model
rfc_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = rfc_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = rfc_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.88      0.90       102
           1       0.89      0.91      0.90       102

    accuracy                           0.90       204
   macro avg       0.90      0.90      0.90       204
weighted avg       0.90      0.90      0.90       204

AUC Score: 0.9480


In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    rfc_clf = RandomForestClassifier(n_estimators=256, max_depth=14, min_samples_split=7, min_samples_leaf=1,
                                 max_features='sqrt',bootstrap=True)
    rfc_clf.fit(X_train, y_train)
    y_pred = rfc_clf.predict(X_test)
    y_proba = rfc_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.9509 [0.9506, 0.9512]
ACC: 0.8995 [0.8988, 0.9001]
F1 : 0.9012 [0.9006, 0.9019]


# Generalization Proved (Non-Clinical): Use combined data for modeling.
## First shuffle the data. SMOTE it. Then, split into 80:20 train test. Train and test the three ensemble models. Report 95% CI of AUC, ACC, F1. Show classification report.  

In [ ]:
df_comb_nonc = df_comb.drop(columns=['chol','restecg','slope', 'oldpeak'])
df_comb_nonc.shape

(1016, 8)

In [ ]:
df_comb_nonc['target'].value_counts()

In [ ]:
df_comb_nonc.head()

,age,sex,cp,trestbps,fbs,thalach,exang,target
0,53,1,4,140,1,155,1,1
1,53,0,4,138,0,160,0,0
2,47,1,4,110,0,118,1,1
3,63,0,2,132,0,130,0,0
4,69,1,4,140,1,109,1,1


In [ ]:
X = df_comb_nonc.drop('target',axis=1)
y = df_comb_nonc['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y,test_size=0.2,random_state=123)


## XGBoost Modeling

In [ ]:
# Hyperparams tuning
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 1.0),
        "use_label_encoder": False,
        "eval_metric": "logloss",
        "objective": "binary:logistic"
    }

    model = XGBClassifier(**params)
    score = cross_val_score(model, X_train, y_train, cv=10, scoring=make_scorer(f1_score)).mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)




In [ ]:
print("Best Hyperparameters:")
print(study.best_params)

Best Hyperparameters:
{'n_estimators': 153, 'max_depth': 3, 'learning_rate': 0.2017724945190854, 'subsample': 0.939903616496261, 'colsample_bytree': 0.948843621246158, 'gamma': 3.5160208313557693, 'reg_alpha': 0.781164043677102, 'reg_lambda': 0.6871466060417619}


In [ ]:
xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False,
    n_estimators = 153, max_depth = 3, learning_rate = 0.2017, subsample = 0.94,
    colsample_bytree = 0.9488, gamma = 3.5160, reg_alpha = 0.7811, reg_lambda = 0.6871
)

xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred))

# Get probabilities for the positive class
y_proba = xgb_clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print("AUC Score:", auc)


              precision    recall  f1-score   support

           0       0.86      0.81      0.83       102
           1       0.82      0.86      0.84       102

    accuracy                           0.84       204
   macro avg       0.84      0.84      0.84       204
weighted avg       0.84      0.84      0.84       204

AUC Score: 0.9055651672433679


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:33:52] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
# 95% CI of accuracy, f1-score, auc score
auc_read = []
acc_read = []
f1_read = []

for i in range(100):
    xgb_clf = xgb.XGBClassifier(objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False,
    n_estimators = 153, max_depth = 3, learning_rate = 0.2017, subsample = 0.94,
    colsample_bytree = 0.9488, gamma = 3.5160, reg_alpha = 0.7811, reg_lambda = 0.6871)

    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)
    y_proba = xgb_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:34:22] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:34:23] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:34:23] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:34:23] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:34:23] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e

AUC: 0.9056 [0.9056, 0.9056]
ACC: 0.8382 [0.8382, 0.8382]
F1 : 0.8421 [0.8421, 0.8421]


## LGboost Modeling

In [ ]:
# Hyperparameter tuning

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "num_leaves": trial.suggest_int("num_leaves", 15, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
    }

    model = LGBMClassifier(**params,verbose=-1)
    score = cross_val_score(model, X_train, y_train, cv=10, scoring=make_scorer(f1_score)).mean()
    return score

# Start Bayesian Optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)




In [ ]:
print('Best hyperparams for LGBoost:')
print(study.best_params)

Best hyperparams for LGBoost:
{'n_estimators': 87, 'max_depth': 3, 'learning_rate': 0.08245411761386941, 'subsample': 0.6705370047240424, 'colsample_bytree': 0.549746921078647, 'num_leaves': 150, 'min_child_samples': 7, 'reg_alpha': 0.9817437590966797, 'reg_lambda': 0.7483023649695567}


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
lgbm_clf = LGBMClassifier(n_estimators=87, max_depth=3, learning_rate=0.0824, subsample=0.6705,
                          colsample_bytree=0.5497, num_leaves=150, min_child_samples=7, reg_alpha=0.9817,
                          reg_lambda=0.7483, verbose=-1)

# 2. Fit the model
lgbm_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = lgbm_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


              precision    recall  f1-score   support

           0       0.84      0.85      0.85       102
           1       0.85      0.84      0.85       102

    accuracy                           0.85       204
   macro avg       0.85      0.85      0.85       204
weighted avg       0.85      0.85      0.85       204

AUC Score: 0.9028


In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    lgbm_clf = LGBMClassifier(n_estimators=87, max_depth=3, learning_rate=0.0824, subsample=0.6705,
                          colsample_bytree=0.5497, num_leaves=150, min_child_samples=7, reg_alpha=0.9817,
                          reg_lambda=0.7483, verbose=-1)
    lgbm_clf.fit(X_train, y_train)
    y_pred = lgbm_clf.predict(X_test)
    y_proba = lgbm_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))


auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.9028 [0.9028, 0.9028]
ACC: 0.8480 [0.8480, 0.8480]
F1 : 0.8473 [0.8473, 0.8473]


## RF Modeling

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
    }

    model = RandomForestClassifier(**params, random_state=42)
    score = cross_val_score(model, X_train, y_train, cv=10, scoring=make_scorer(f1_score)).mean()
    return score

# Start the study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)


In [ ]:
# Show results
print("Best Hyperparameters for RandomForest:")
print(study.best_params)

Best Hyperparameters for RandomForest:
{'n_estimators': 110, 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'bootstrap': True}


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize the model
rfc_clf = RandomForestClassifier(n_estimators=110, max_depth=20, min_samples_split=4, min_samples_leaf=8,
                                 max_features='sqrt',bootstrap=True)

# 2. Fit the model
rfc_clf.fit(X_train, y_train)

# 3. Predict class labels
y_pred = rfc_clf.predict(X_test)

# 4. Predict probabilities (for AUC)
y_proba = rfc_clf.predict_proba(X_test)[:, 1]

# 5. Classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# 6. AUC Score
auc = roc_auc_score(y_test, y_proba)
print(f"AUC Score: {auc:.4f}")


Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.82      0.85       102
           1       0.83      0.88      0.86       102

    accuracy                           0.85       204
   macro avg       0.85      0.85      0.85       204
weighted avg       0.85      0.85      0.85       204

AUC Score: 0.9135


In [ ]:
auc_read = []
acc_read = []
f1_read = []

for i in range(100):

    rfc_clf = RandomForestClassifier(n_estimators=110, max_depth=20, min_samples_split=4, min_samples_leaf=8,
                                 max_features='sqrt',bootstrap=True)
    rfc_clf.fit(X_train, y_train)
    y_pred = rfc_clf.predict(X_test)
    y_proba = rfc_clf.predict_proba(X_test)[:, 1]

    auc_read.append(roc_auc_score(y_test, y_proba))
    acc_read.append(accuracy_score(y_test, y_pred))
    f1_read.append(f1_score(y_test, y_pred))

auc_mean, auc_low, auc_high = get_ci(auc_read)
acc_mean, acc_low, acc_high = get_ci(acc_read)
f1_mean, f1_low, f1_high = get_ci(f1_read)

print(f"AUC: {auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}]")
print(f"ACC: {acc_mean:.4f} [{acc_low:.4f}, {acc_high:.4f}]")
print(f"F1 : {f1_mean:.4f} [{f1_low:.4f}, {f1_high:.4f}]")

AUC: 0.9085 [0.9081, 0.9089]
ACC: 0.8498 [0.8488, 0.8508]
F1 : 0.8536 [0.8527, 0.8544]


## Next Stage: Report on bias metrics, and SMOTE the theoretical regional dataset to balance diseased and non-diseased.
https://colab.research.google.com/drive/1Zx0GTOdJYghEAIGale2AMWQAHmB1aaRH?usp=sharing